# 🧩 Layer 1: Provincial Typology (Clustering)

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = Path('cluster_output')
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
# Load Data using local path
df = pd.read_csv('../data_bps_datmin.csv')
print('Dataset Loaded!')


In [ ]:
# Aggregate Data by Province
df['tpt'] = (df['TPT - Februari'] + df['TPT - Agustus']) / 2
df['tpak'] = (df['TPAK - Februari'] + df['TPAK - Agustus']) / 2
df['poverty_rate'] = (df['Persentase Penduduk Miskin - Maret'] + df['Persentase Penduduk Miskin - September']) / 2
df['hdi'] = df['Indeks Pembangunan Manusia']
features = ['tpt', 'tpak', 'poverty_rate', 'hdi']
df_cluster = df.groupby('PROVINSI')[features].mean()


In [ ]:
# Standardization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster)
df_scaled = pd.DataFrame(X_scaled, columns=features, index=df_cluster.index)


In [ ]:
# Feature Correlation
corr = df_scaled.corr()
fig = px.imshow(corr, text_auto=".2f", aspect="auto", color_continuous_scale="RdBu_r", zmin=-1, zmax=1)
fig.update_layout(title="Feature Correlation Matrix", width=800, height=800)
fig.write_json(OUTPUT_DIR / 'plot_feature_correlation.json')
fig.show()


In [ ]:
# Determine Optimal K
K = range(2, 8)
inertias = []
sil_scores = []
for k in K:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig_elbow = go.Figure()
fig_elbow.add_trace(go.Scatter(x=list(K), y=inertias, mode='lines+markers', name='Inertia'))
fig_elbow.update_layout(title="Elbow Method for Optimal k", xaxis_title="k", yaxis_title="Inertia", width=800, height=500)
fig_elbow.write_json(OUTPUT_DIR / 'plot_optimum_clusters_elbow.json')
fig_elbow.show()

fig_sil = go.Figure()
fig_sil.add_trace(go.Scatter(x=list(K), y=sil_scores, mode='lines+markers', line=dict(color='orange')))
fig_sil.update_layout(title="Silhouette Score for Optimal k", xaxis_title="k", yaxis_title="Silhouette Score", width=800, height=500)
fig_sil.write_json(OUTPUT_DIR / 'plot_optimum_clusters_silhouette.json')
fig_sil.show()


In [ ]:
# Final K-Means Clustering (k=2)
km_opt = KMeans(n_clusters=2, random_state=42)
df_cluster['cluster'] = km_opt.fit_predict(X_scaled).astype(str)
df_cluster['cluster_label'] = df_cluster['cluster'].map({'0': 'Cluster 0', '1': 'Cluster 1'})
df_cluster.to_csv(OUTPUT_DIR / 'output_province_clusters.csv')


In [ ]:
# PCA Projection
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)
df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2', 'PC3'], index=df_cluster.index)
df_pca['cluster'] = df_cluster['cluster_label']
df_pca = df_pca.reset_index()

fig_2d = px.scatter(df_pca, x='PC1', y='PC2', color='cluster', hover_data=['PROVINSI'])
fig_2d.update_layout(title="K-Means Clustering (2D PCA Projection)", width=800, height=600)
fig_2d.write_json(OUTPUT_DIR / 'plot_kmeans_pca_2d.json')
fig_2d.show()

fig_3d = px.scatter_3d(df_pca, x='PC1', y='PC2', z='PC3', color='cluster', hover_data=['PROVINSI'])
fig_3d.update_layout(title="K-Means Clustering (3D PCA Projection)", width=900, height=700)
fig_3d.write_json(OUTPUT_DIR / 'plot_kmeans_pca_3d.json')
fig_3d.show()
